In [1]:
import requests

def get_product_data(barcode):
    """
    Obtiene datos de un producto desde la API v2 de Open Food Facts.
    """
    # 1. Configurar el User-Agent (REQUISITO CRÍTICO DE LA API)
    headers = {
        'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'
    }
    
    # 2. Construir la URL de la API v2
    url = f"https://world.openfoodfacts.org/api/v2/product/{barcode}.json"
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            if data.get('status') == 1: # 1 significa producto encontrado
                product = data['product']
                
                # 3. Extraer solo lo que necesitamos para el ML
                result = {
                    'nombre': product.get('product_name', 'Desconocido'),
                    'nutriscore': product.get('nutriscore_grade', 'N/A'),
                    'nova': product.get('nova_group', 'N/A'),
                    # Estos son los tags de aditivos ya procesados por su IA
                    'aditivos_lista': product.get('additives_tags', []),
                    'conteo_aditivos': len(product.get('additives_tags', [])),
                    'ingredientes': product.get('ingredients_text', ''),
                    'categorias': product.get('categories_hierarchy', [])
                }
                return result
            else:
                return "Producto no encontrado en la base de datos."
        else:
            return f"Error en la petición: {response.status_code}"
            
    except Exception as e:
        return f"Error de conexión: {e}"

# Ejemplo de uso con un código de barras real (ej. Nutella)
# print(get_product_data("3017620422003"))

In [2]:
get_product_data("8410076801180")

{'nombre': 'Crunchy oats&honey',
 'nutriscore': 'd',
 'nova': 4,
 'aditivos_lista': ['en:e322', 'en:e500', 'en:e500ii'],
 'conteo_aditivos': 3,
 'ingredientes': 'Avoine complète (60%), sucre, huiles végétales (tournesol, colza en proportions variables), miel (3%), sel de cuisine, mélasse, émulsifiant (lécithines), poudre à lever (bicarbonate de sodium).',
 'categorias': ['en:plant-based-foods-and-beverages',
  'en:plant-based-foods',
  'en:snacks',
  'en:cereals-and-potatoes',
  'en:sweet-snacks',
  'en:bars',
  'en:cereal-bars']}

In [4]:
import requests
import pandas as pd

def generar_maestro_aditivos():
    url = "https://world.openfoodfacts.org/data/taxonomies/additives.json"
    headers = {'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'}
    
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        
        lista_limpia = []
        for key, value in data.items():
            # Solo nos interesan los que tienen nombre en español o inglés
            nombre = value.get('name', {}).get('en', value.get('name', {}).get('en', 'Desconocido'))
            
            lista_limpia.append({
                'id': key,
                'nombre': nombre
            })
        
        df = pd.DataFrame(lista_limpia)
        df.to_csv('./maestro_aditivos.csv', index=False)
        print(f"¡Hecho! Tienes {len(df)} aditivos registrados.")
        return df

# Ejecutar una vez para tener tu archivo local
# df_aditivos = generar_maestro_aditivos()

In [5]:
df_aditivos = generar_maestro_aditivos()

¡Hecho! Tienes 681 aditivos registrados.


In [6]:
df_aditivos.head()

,id,nombre
0,en:e333ii,E333ii - Dicalcium citrate
1,en:e321,E321 - Butylated hydroxytoluene
2,en:e345,E345 - Magnesium citrate
3,en:e303,E303 - Potassium ascorbate
4,en:e1104,E1104 - lipase
